In [1]:
# %pip install --user "numpy<2" "scipy>=1.11,<1.14" "pandas==2.2.3" "tensorflow[and-cuda]==2.15.1" \
#     "opencv-python-headless<4.12" "scikit-image<0.25" girder-client openslide-python openslide-bin openpyxl matplotlib

In [2]:
import numpy, tensorflow as tf, pandas, cv2, openslide, girder_client
print("numpy", numpy.__version__, "| tf", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

2026-11-12 14:24:37.033917: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-12 14:24:37.393767: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-12 14:24:37.393829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-12 14:24:37.470207: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-12 14:24:37.624806: I tensorflow/core/platform/cpu_feature_guar

numpy 1.26.4 | tf 2.15.1 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [37]:
"""
mvi_pipeline.ipynb: iptcr scoring for CD34-LCA (CD34 red / CD45 brown) kidney biopsies on Athena.
 
For every slide (one ITEM_ID, or every slide in FOLDER_IDS):
  1. downloads the slide from Athena to WORK_DIR
  2. finds tissue (background tiles are skipped)
  3. predicts glomeruli + arteries (TransUNet) and peritubular capillaries (PTC U-Net) on
     overlapping 224 px tiles at the resolution the models were trained at; overlaps are averaged
  4. finds brown DAB (CD45 leukocytes) with a 3-stain colour deconvolution (hematoxylin / red / DAB),
     so red CD34 is NOT counted as brown
  5. PTC objects: drops tiny specks, PTCs inside glomeruli/arteries, PTCs inside large DAB aggregates,
     and (optional) PTCs outside an ROI layer such as "cortex"
  6. iptcr = PTCs touching >= 1 leukocyte / PTCs counted       (also at 3 / 8 / 15 % area overlap)
  7. saves: results CSV, Aperio XML with named layers, QC images; optionally uploads layers to Athena
 
ACTION = "check"  -> read-only: lists the slides and which ones already have iptcr results
ACTION = "run"    -> runs the pipeline (slides already done are skipped unless RERUN = True)
                     results stay on HiPerGator (CSV, XML, QC images); nothing is written to Athena
                     unless UPLOAD_TO_ATHENA / WRITE_METADATA are True
ACTION = "upload" -> sends the AI layers of slides that already ran (their saved XML) to the slides on
                     Athena, without running the models again. Asks for YES first.
 
Run on a HiPerGator GPU with run_mvi.sh.
"""
import os
import re
import csv
import json
import math
import time
import argparse
import traceback
import xml.etree.ElementTree as ET
 
import numpy as np
import cv2 as cv

In [92]:
# =================================================================== SETTINGS
ACTION = "run"                       # check | run
 
# ---- Athena
INSTANCE = "athena"                  # athena | devathena
FOLDER_IDS = [
    "FLODER-ID",
]
RECURSIVE = True
# TEST ON ONE SLIDE: paste its item ID. Comment out the 2nd line to run the whole folder.
ITEM_ID = None
# ITEM_ID = "ITEM-ID-IF-NEEDED"
 
# ---- where things go on HiPerGator
WORK_DIR = "/home/pvempati/MVI_test/mvi_iptcr"
DELETE_SLIDES_AFTER = True           # delete each downloaded slide after it is processed
RERUN = True                        # True = process slides again even if results exist
 
# ---- models
PTC_MODEL_PATH = "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/PTC_IPTCR/best_model.h5"
GLOM_MODEL_PATH = ("/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Model_Development/"
                   "Glom_Model_UNet/transunet_multiclass_denseNet_dice_newdata.h5")
GLOM_READ_SIZE = 512                 # area read per glom tile, in analysis pixels (0.91 um/px):
                                     #   512 = model sees tissue at its true size (a whole glomerulus fits)
                                     #   1024 = zoomed out 2x (try if glomeruli still come out in pieces)
GLOM_OVERLAP = 0.5                   # glom tiles overlap 50%; predictions are blended
GLOM_FILL_HOLES = True               # fill holes inside each glomerulus/artery (one solid shape each)
GLOM_CLASSES = {"background": 0, "glomerulus": 1, "artery": 2}
USE_GLOM_MODEL = True                # False = skip glom/artery (PTCs inside glomeruli are then kept)
 
# ---- resolution: the models were trained on NDPI level 2 = 0.2281 um/px x 4
TRAIN_MPP = 0.2281 * 4               # micrometres per pixel the models expect
FALLBACK_DOWNSAMPLE = 4.0            # used only if a slide has no pixel size in its metadata
 
# ---- tiling
TILE = 224
OVERLAP = 0.5                        # 50% overlap; predictions are averaged where tiles overlap
BATCH_SIZE = 16
TISSUE_MIN_FRACTION = 0.05           # skip tiles with < 5% tissue
 
# ---- optional ROI: only count PTCs inside an Athena annotation layer with this name (e.g. "cortex")
ROI_LAYER_NAME = None
 
# ---- PTC / glom / artery post-processing
PTC_THRESHOLD = 0.5
PTC_FILL_HOLES_UM2 = 60.0            # fill holes (lumens) smaller than this inside predicted PTCs
PTC_MIN_AREA_UM2 = 4.0               # drop smaller PTC specks
PTC_MAX_GLOM_ARTERY_OVERLAP = 0.10   # drop a PTC if > 10% of it lies in a glomerulus/artery
GLOM_MIN_AREA_UM2 = 1250.0           # = 1500 px in the old notebook
ARTERY_MIN_AREA_UM2 = 415.0          # = 500 px in the old notebook
 
# ---- DAB (CD45 leukocytes) via hematoxylin / red / DAB colour deconvolution
STAIN_VECTORS = {                    # optical-density vectors (Ruifrok & Johnston; ImageJ Colour Deconvolution)
    "hematoxylin": [0.650, 0.704, 0.286],
    "red":         [0.214, 0.851, 0.478],   # CD34 red chromogen
    "dab":         [0.268, 0.570, 0.776],
}
DAB_THRESHOLD = 0.25                 # brown pixel if DAB > this, DAB > red, and DAB > 1.2 x hematoxylin
DAB_HUE_RANGE = (4, 30)
DAB_MIN_AREA_UM2 = 8.0               # smaller brown specks are noise
DAB_AGGREGATE_AREA_UM2 = 1500.0      # larger brown blobs are aggregates, not single leukocytes
AGGREGATE_SURROUND_FRACTION = 0.5    # drop a PTC if > 50% of the pixels around it are a DAB aggregate
# "dab_inflamed" layer = dense brown INFILTRATE regions, as in the approved data (orange regions)
INFILTRATE_MERGE_PX = 5              # brown cells closer than ~this many pixels are merged into one region
INFILTRATE_MIN_AREA_UM2 = 24000.0    # regions at least this large = dab_inflamed (matches the approved data)

# ---- inflamed PTC rule
PTC_RING_PX = 1                      # also count leukocytes touching the PTC edge (marginated)
MIN_LEUKO_PIXELS = 3                 # "touching >= 1 leukocyte" = at least this many brown pixels
IPTCR_PERCENTS = [3, 8, 15]          # also report iptcr with the old "% of PTC area brown" rules
 
# ---- outputs
SAVE_XML = True
SAVE_QC_IMAGES = True
SAVE_EXCEL = True                    # iptcr_results.xlsx (one row per slide) + tables/<slide>.xlsx (one row per PTC)
N_QC_CROPS = 4
UPLOAD_TO_ATHENA = True             # True = add the AI layers to the slide on Athena
REPLACE_AI_LAYERS = True             # when uploading, first delete this script's old layers on that slide
WRITE_METADATA = False               # True = also save AI_iptcr etc. as slide metadata on Athena
LAYER_PREFIX = "AI "
ASSUME_YES = False                   # True (or --yes) = upload without asking for YES (batch jobs)
 
# ---- "check": a CSV that already has iptcr scores (e.g. the older Scores.csv), optional
REFERENCE_CSV = None                 # e.g. "/blue/<group>/<gatorlink>/Scores.csv"
REFERENCE_ID_COLUMN = None           # column with the slide name/ID; None = first column
REFERENCE_SCORE_COLUMNS = None       # None = every column whose name contains "iptcr"


In [93]:
# =================================================================== internals
LAYERS = {   # same 5 layers as the approved data; name -> (Aperio LineColor (BGR int), HistomicsUI rgb)
    "glomerulus":   (65280,    "rgb(0,255,0)"),
    "artery":       (16776960, "rgb(0,255,255)"),
    "ptc":          (16711935, "rgb(255,0,255)"),
    "dab_inflamed": (255,      "rgb(255,0,0)"),       # leukocytes inside / touching a counted PTC
    "dab_normal":   (65535,    "rgb(255,255,0)"),     # all other brown (leukocytes elsewhere + aggregates)
}
ID_PATTERN = re.compile(r"JHH[\s_-]*DS[\s_-]*0*(\d+)", re.IGNORECASE)
RESULT_FIELDS = ["slide", "slide_key", "item_id", "folder", "status", "iptcr", "n_ptc_counted",
                 "n_ptc_inflamed"] + [f"iptcr_pct{p}" for p in IPTCR_PERCENTS] + [
                 "n_ptc_detected", "n_ptc_excl_glom_artery", "n_ptc_excl_dab_aggregate",
                 "n_ptc_excl_roi", "n_glomeruli", "n_arteries", "n_leukocytes", "n_dab_aggregates",
                 "tissue_mm2", "mpp_analysis", "downsample", "minutes", "error"]
 
 
def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)
 
 
def slide_key(name):
    """JHH_DS_12 from 'JHH_DS_12 CD34LCA.ndpi'; otherwise a normalised name (S15-69213CD34LCA -> s1569213)."""
    m = ID_PATTERN.search(name)
    if m:
        return f"JHH_DS_{int(m.group(1))}"
    base = os.path.basename(str(name))
    while os.path.splitext(base)[1]:
        base = os.path.splitext(base)[0]
    base = re.sub(r"\s*-\s*\d{4}-\d{2}-\d{2}.*$", "", base)          # " - 2016-12-12 16.55.44"
    base = re.sub(r"cd34\s*[-_ ]?\s*(lca|cd45)", "", base, flags=re.I)  # stain suffix
    return re.sub(r"[^a-z0-9]", "", base.lower())
 
 

In [94]:
# ------------------------------------------------------------------- Athena
def connect():
    import girder_client
    url = {"athena": "https://athena.rc.ufl.edu/api/v1/",
           "devathena": "https://devathena.rc.ufl.edu/api/v1/"}[INSTANCE]
    key = None
    try:                                         # same credentials as Metadata.py, if common.py is here
        import common
        url, key, _, _, _ = common.urlAndCredentials(INSTANCE)
    except Exception:
        key = os.environ.get("ATHENA_API_KEY" if INSTANCE == "athena" else "DEVATHENA_API_KEY")
    gc = girder_client.GirderClient(apiUrl=url)
    if key:
        gc.authenticate(apiKey=key)
    else:
        gc.authenticate(interactive=True)
    return gc
 
 
def list_slides(gc, item_id=None, folder_ids=None):
    if item_id:
        it = gc.getItem(item_id)
        return [(it, "")]
    out = []
 
    def walk(fid, path):
        for it in gc.listItem(fid):
            if "largeImage" in it:
                out.append((it, path))
        if RECURSIVE:
            for sub in gc.listFolder(fid, parentFolderType="folder"):
                walk(sub["_id"], f"{path}/{sub['name']}")
 
    for fid in folder_ids:
        walk(fid, gc.getFolder(fid)["name"])
    return out
 
 
def download_slide(gc, item):
    li = item.get("largeImage", {})
    file_id = li.get("originalId") or li.get("fileId")     # prefer the original .ndpi/.svs
    f = gc.getFile(file_id)
    os.makedirs(os.path.join(WORK_DIR, "slides"), exist_ok=True)
    path = os.path.join(WORK_DIR, "slides", f"{item['_id']}_{f['name']}")
    if os.path.exists(path) and os.path.getsize(path) == f.get("size"):
        return path
    log(f"   downloading {f['name']} ({f.get('size', 0) / 1e6:.0f} MB)")
    gc.downloadFile(file_id, path)
    return path
 
 
def roi_polygons(gc, item_id):
    """ROI polygons (level-0 coordinates) from the Athena layer named ROI_LAYER_NAME."""
    polys = []
    for a in gc.get("annotation", parameters={"itemId": item_id, "limit": 0}):
        if a["annotation"].get("name", "").strip().lower() != ROI_LAYER_NAME.strip().lower():
            continue
        full = gc.get(f"annotation/{a['_id']}")
        for el in full["annotation"].get("elements", []):
            if el.get("type") == "polygon":
                polys.append(np.array([p[:2] for p in el["points"]], dtype=np.float64))
            elif el.get("type") == "rectangle":
                cx, cy = el["center"][:2]
                w, h = el["width"], el["height"]
                polys.append(np.array([[cx - w / 2, cy - h / 2], [cx + w / 2, cy - h / 2],
                                       [cx + w / 2, cy + h / 2], [cx - w / 2, cy + h / 2]]))
    return polys
 
 
def file_base(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", os.path.splitext(name)[0])
 
 
def read_xml_layers(path):
    """Layers saved by save_xml -> {layer name without prefix: [polygons]}."""
    layers = {}
    for ann in ET.parse(path).getroot().findall("Annotation"):
        name = ann.get("Name", "")
        name = name[len(LAYER_PREFIX):] if name.startswith(LAYER_PREFIX) else name
        if name not in LAYERS:
            continue
        polys = []
        for reg in ann.iter("Region"):
            pts = [(float(v.get("X")), float(v.get("Y"))) for v in reg.iter("Vertex")]
            if len(pts) > 3:
                polys.append(pts[:-1])                     # last vertex repeats the first
        layers[name] = polys
    return layers
 
 
def upload_layers(gc, item_id, layers_level0):
    if REPLACE_AI_LAYERS:
        for a in gc.get("annotation", parameters={"itemId": item_id, "limit": 0}):
            if a["annotation"].get("name", "").startswith(LAYER_PREFIX):     # only this script's layers
                gc.delete(f"annotation/{a['_id']}")
    for name, polys in layers_level0.items():
        if not polys:
            continue
        rgb = LAYERS[name][1]
        elements = [{"type": "polyline", "closed": True, "lineColor": rgb, "lineWidth": 2,
                     "fillColor": rgb.replace("rgb(", "rgba(").replace(")", ",0.15)"),
                     "points": [[float(x), float(y), 0] for x, y in p]} for p in polys]
        gc.post("annotation", parameters={"itemId": item_id},
                json={"name": LAYER_PREFIX + name, "elements": elements})
 
 

In [95]:
# ------------------------------------------------------------------- slide reading
class SlideReader:
    """Reads tiles at the models' training resolution (TRAIN_MPP), whatever the scanner/format."""
 
    def __init__(self, path):
        import openslide
        self.s = openslide.OpenSlide(path)
        props = self.s.properties
        mpp0 = props.get("openslide.mpp-x")
        if mpp0:
            self.mpp0 = float(mpp0)
            self.ds = TRAIN_MPP / self.mpp0              # level-0 pixels per analysis pixel
        else:
            self.mpp0 = None
            self.ds = FALLBACK_DOWNSAMPLE
            log(f"   WARNING: no pixel size in slide metadata, using downsample {FALLBACK_DOWNSAMPLE}")
        self.mpp = TRAIN_MPP if self.mpp0 else None
        self.level = self.s.get_best_level_for_downsample(self.ds + 1e-6)
        self.level_ds = self.s.level_downsamples[self.level]
        W0, H0 = self.s.dimensions
        self.W, self.H = int(W0 / self.ds), int(H0 / self.ds)
 
    def read(self, x, y, size):
        """RGB uint8 (size x size) at analysis resolution; area outside the slide is white."""
        n = int(math.ceil(size * self.ds / self.level_ds))
        reg = np.asarray(self.s.read_region((int(round(x * self.ds)), int(round(y * self.ds))),
                                            self.level, (n, n)), dtype=np.float32)
        a = reg[..., 3:4] / 255.0
        rgb = (reg[..., :3] * a + 255.0 * (1 - a)).astype(np.uint8)       # transparent -> white
        if n != size:
            rgb = cv.resize(rgb, (size, size), interpolation=cv.INTER_AREA)
        return rgb
 
    def thumbnail(self, max_side=2048):
        """(RGB thumbnail, analysis pixels per thumbnail pixel)."""
        th = np.asarray(self.s.get_thumbnail((max_side, max_side)).convert("RGB"))
        return th, self.W / th.shape[1]
 
 

In [96]:
# ------------------------------------------------------------------- image helpers
_STAIN_CONV = None
 
 
def stain_channels(rgb):
    global _STAIN_CONV
    if _STAIN_CONV is None:
        M = np.array([STAIN_VECTORS["hematoxylin"], STAIN_VECTORS["red"], STAIN_VECTORS["dab"]], float)
        M /= np.linalg.norm(M, axis=1, keepdims=True)
        _STAIN_CONV = np.linalg.inv(M)
    od = -np.log10(np.maximum(rgb.astype(np.float32), 1.0) / 255.0)
    return np.maximum(od.reshape(-1, 3) @ _STAIN_CONV, 0).reshape(rgb.shape)
 
 
def dab_mask(rgb):
    s = stain_channels(rgb)
    h, red, dab = s[..., 0], s[..., 1], s[..., 2]
    hue = cv.cvtColor(rgb, cv.COLOR_RGB2HSV)[..., 0]
    return ((dab > DAB_THRESHOLD) & (dab > red) & (dab > 1.2 * h)
            & (hue >= DAB_HUE_RANGE[0]) & (hue <= DAB_HUE_RANGE[1]))
 
 
def tissue_mask(thumb):
    hsv = cv.cvtColor(thumb, cv.COLOR_RGB2HSV)
    sat = hsv[..., 1]
    t, _ = cv.threshold(sat, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    m = (sat > max(t, 15)) & (hsv[..., 2] > 30)                         # coloured and not black
    m = cv.morphologyEx(m.astype(np.uint8), cv.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    return cv.morphologyEx(m, cv.MORPH_OPEN, np.ones((3, 3), np.uint8)).astype(bool)
 
 
_WEIGHTS = {}


def tile_weight(size=TILE):
    """Centre-weighted blending: tile centres count most, edges (less reliable) least."""
    if size not in _WEIGHTS:
        w = np.sin(np.linspace(0, np.pi, size + 2)[1:-1]) ** 2
        _WEIGHTS[size] = np.maximum(np.outer(w, w), 0.02).astype(np.float32)
    return _WEIGHTS[size]


def positions(length, tile=TILE, overlap=OVERLAP):
    stride = max(1, int(tile * (1 - overlap)))
    if length < tile:
        return [0] if length > 0 else []
    p = list(range(0, length - tile + 1, stride))
    if p[-1] != length - tile:
        p.append(length - tile)                                          # cover the last edge
    return p


def tissue_tiles(W, H, tis, f, tile, overlap):
    out = []
    for y in positions(H, tile, overlap):
        for x in positions(W, tile, overlap):
            ty0, ty1 = int(y / f), max(int((y + tile) / f), int(y / f) + 1)
            tx0, tx1 = int(x / f), max(int((x + tile) / f), int(x / f) + 1)
            if tis[ty0:ty1, tx0:tx1].mean() >= TISSUE_MIN_FRACTION:
                out.append((x, y))
    return out
 
def remove_small(mask, min_px):
    n, lab, st, _ = cv.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = st[1:, cv.CC_STAT_AREA] >= min_px
    return keep[lab]
 
 
def contours_of(mask, scale):
    cs, _ = cv.findContours(mask.astype(np.uint8), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cs:
        if len(c) < 3:
            continue
        c = cv.approxPolyDP(c, 0.7, True)
        if len(c) >= 3:
            out.append(c[:, 0, :].astype(np.float64) * scale)
    return out
 
 

In [97]:
# ------------------------------------------------------------------- models
def load_models():
    import tensorflow as tf
    for g in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception:
            pass
    log(f"GPUs: {tf.config.list_physical_devices('GPU') or 'none (CPU will be slow)'}")
    ptc = tf.keras.models.load_model(PTC_MODEL_PATH, compile=False)
    glom = tf.keras.models.load_model(GLOM_MODEL_PATH, compile=False) if USE_GLOM_MODEL else None
    return ptc, glom
 
 
def predict(model, batch):
    return np.asarray(model.predict(batch, batch_size=len(batch), verbose=0))
 
 

In [98]:
# ------------------------------------------------------------------- one slide
def analyse(reader, ptc_model, glom_model, roi_level0=None, qc_prefix=None):
    W, H, ds = reader.W, reader.H, reader.ds
    mpp = reader.mpp or (reader.mpp0 or TRAIN_MPP / FALLBACK_DOWNSAMPLE) * ds
    px_area = mpp * mpp                                                  # um^2 per analysis pixel

    thumb, f = reader.thumbnail()
    tis = tissue_mask(thumb)
    roi_mask = None
    if roi_level0 is not None:
        roi_mask = np.zeros((H, W), np.uint8)
        cv.fillPoly(roi_mask, [np.round(p / ds).astype(np.int32) for p in roi_level0], 1)
        roi_mask = roi_mask.astype(bool)
        roi_thumb = cv.resize(roi_mask.astype(np.uint8), (thumb.shape[1], thumb.shape[0]),
                              interpolation=cv.INTER_NEAREST).astype(bool)
        tis &= roi_thumb

    tiles = tissue_tiles(W, H, tis, f, TILE, OVERLAP)
    log(f"   analysis size {W}x{H} px at {mpp:.3f} um/px, {len(tiles)} PTC tiles")

    # ---- pass 1: PTC model (224 px tiles, true scale) + DAB
    ptc_sum = np.zeros((H, W), np.float32)
    wsum = np.zeros((H, W), np.float32)
    wt = tile_weight(TILE)
    dab = np.zeros((H, W), bool)
    for b in range(0, len(tiles), BATCH_SIZE):
        chunk = tiles[b:b + BATCH_SIZE]
        imgs = np.stack([reader.read(x, y, TILE) for x, y in chunk])
        p = predict(ptc_model, imgs.astype(np.float32) / 255.0)[..., 0]
        for i, (x, y) in enumerate(chunk):
            ptc_sum[y:y + TILE, x:x + TILE] += p[i] * wt
            wsum[y:y + TILE, x:x + TILE] += wt
            dab[y:y + TILE, x:x + TILE] = dab_mask(imgs[i])
        if (b // BATCH_SIZE) % 50 == 0:
            log(f"   PTC tiles {min(b + BATCH_SIZE, len(tiles))}/{len(tiles)}")
    seen = wsum > 0
    ptc = np.zeros((H, W), bool)
    ptc[seen] = (ptc_sum[seen] / wsum[seen]) > PTC_THRESHOLD
    del ptc_sum, wsum
    if PTC_FILL_HOLES_UM2 > 0:
        from skimage.morphology import remove_small_holes
        ptc = remove_small_holes(ptc, area_threshold=int(PTC_FILL_HOLES_UM2 / px_area))

    # ---- pass 2: glom/artery model on its OWN larger tiles, so whole glomeruli fit in one tile
    glom = np.zeros((H, W), bool)
    art = np.zeros((H, W), bool)
    if glom_model is not None:
        gt = min(GLOM_READ_SIZE, W, H)
        gtiles = tissue_tiles(W, H, tis, f, gt, GLOM_OVERLAP)
        log(f"   glom tiles: {len(gtiles)} x {gt} px")
        gsum = np.zeros((H, W, len(GLOM_CLASSES)), np.float32)
        gw = np.zeros((H, W), np.float32)
        gwt = tile_weight(gt)
        bs = max(1, BATCH_SIZE // 4)
        for b in range(0, len(gtiles), bs):
            chunk = gtiles[b:b + bs]
            imgs = [reader.read(x, y, gt) for x, y in chunk]
            g_in = np.stack([im if gt == GLOM_INPUT_SIZE else
                             cv.resize(im, (GLOM_INPUT_SIZE, GLOM_INPUT_SIZE), interpolation=cv.INTER_AREA)
                             for im in imgs])
            g = predict(glom_model, g_in.astype(np.float32) / 255.0)
            for i, (x, y) in enumerate(chunk):
                gi = g[i].astype(np.float32)
                if gi.shape[0] != gt:
                    gi = cv.resize(gi, (gt, gt), interpolation=cv.INTER_LINEAR)
                gsum[y:y + gt, x:x + gt] += gi * gwt[..., None]
                gw[y:y + gt, x:x + gt] += gwt
        cls = np.argmax(gsum, axis=-1).astype(np.uint8)
        cls[gw == 0] = GLOM_CLASSES["background"]
        del gsum, gw
        k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (7, 7))
        for name in ("glomerulus", "artery"):
            m = (cls == GLOM_CLASSES[name]).astype(np.uint8)
            m = cv.morphologyEx(cv.morphologyEx(m, cv.MORPH_CLOSE, k), cv.MORPH_OPEN, k).astype(bool)
            if GLOM_FILL_HOLES:
                from scipy.ndimage import binary_fill_holes
                m = binary_fill_holes(m)
            min_um2 = GLOM_MIN_AREA_UM2 if name == "glomerulus" else ARTERY_MIN_AREA_UM2
            m = remove_small(m, min_um2 / px_area)
            if name == "glomerulus":
                glom = m
            else:
                art = m & ~glom
        del cls

    # ---- PTC objects
    n, lab, st, cen = cv.connectedComponentsWithStats(ptc.astype(np.uint8), connectivity=8)
    area = st[:, cv.CC_STAT_AREA].astype(np.float64)
    valid = np.zeros(n, bool)
    valid[1:] = area[1:] * px_area >= PTC_MIN_AREA_UM2
    ga = np.bincount(lab.ravel(), weights=(glom | art).ravel(), minlength=n)
    excl_ga = valid & (ga / np.maximum(area, 1) > PTC_MAX_GLOM_ARTERY_OVERLAP)

    # ---- DAB objects: leukocytes vs aggregates
    dn, dlab, dst, _ = cv.connectedComponentsWithStats(dab.astype(np.uint8), connectivity=8)
    darea = dst[:, cv.CC_STAT_AREA] * px_area
    is_leuko = (darea >= DAB_MIN_AREA_UM2) & (darea <= DAB_AGGREGATE_AREA_UM2)
    is_aggr = darea > DAB_AGGREGATE_AREA_UM2
    is_leuko[0] = is_aggr[0] = False
    leuko = is_leuko[dlab]
    aggr = is_aggr[dlab]
    del dab

    cy = np.clip(np.round(cen[:, 1]).astype(int), 0, H - 1)
    cx = np.clip(np.round(cen[:, 0]).astype(int), 0, W - 1)
    # pixels around each PTC (2-px ring): a PTC embedded in a brown aggregate is surrounded by it
    from scipy.ndimage import grey_dilation
    lab_r = grey_dilation(lab, size=5)
    lab_r[lab > 0] = 0
    ring_area = np.bincount(lab_r.ravel(), minlength=n)
    ring_aggr = np.bincount(lab_r.ravel(), weights=aggr.ravel(), minlength=n)
    del lab_r
    excl_aggr = valid & ~excl_ga & (ring_aggr / np.maximum(ring_area, 1) > AGGREGATE_SURROUND_FRACTION)
    excl_roi = np.zeros(n, bool)
    if roi_mask is not None:
        excl_roi = valid & ~excl_ga & ~excl_aggr & ~roi_mask[cy, cx]
    counted = valid & ~excl_ga & ~excl_aggr & ~excl_roi

    # ---- inflamed: leukocyte pixels inside the PTC or on a 1-px ring around it
    if PTC_RING_PX > 0:
        lab_d = grey_dilation(lab, size=2 * PTC_RING_PX + 1)
        lab_d = np.where(lab > 0, lab, lab_d)
    else:
        lab_d = lab
    leuko_ring = np.bincount(lab_d.ravel(), weights=leuko.ravel(), minlength=n)
    leuko_in = np.bincount(lab.ravel(), weights=leuko.ravel(), minlength=n)
    # dab_inflamed = dense brown infiltrate regions (as in the approved data); dab_normal = all other brown
    brown = (is_leuko | is_aggr)[dlab]
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (INFILTRATE_MERGE_PX, INFILTRATE_MERGE_PX))
    merged = cv.morphologyEx(brown.astype(np.uint8), cv.MORPH_CLOSE, k)
    from scipy.ndimage import binary_fill_holes
    dab_infl = remove_small(binary_fill_holes(merged), INFILTRATE_MIN_AREA_UM2 / px_area)
    dab_norm = brown & ~dab_infl
    del lab_d, dlab, merged
    inflamed = counted & (leuko_ring >= MIN_LEUKO_PIXELS)
    pct = 100 * leuko_in / np.maximum(area, 1)

    n_count = int(counted.sum())
    res = {
        "iptcr": inflamed.sum() / n_count if n_count else float("nan"),
        "n_ptc_counted": n_count, "n_ptc_inflamed": int(inflamed.sum()),
        "n_ptc_detected": int(valid.sum()), "n_ptc_excl_glom_artery": int(excl_ga.sum()),
        "n_ptc_excl_dab_aggregate": int(excl_aggr.sum()), "n_ptc_excl_roi": int(excl_roi.sum()),
        "n_glomeruli": cv.connectedComponents(glom.astype(np.uint8))[0] - 1,
        "n_arteries": cv.connectedComponents(art.astype(np.uint8))[0] - 1,
        "n_leukocytes": int(is_leuko.sum()), "n_dab_aggregates": int(is_aggr.sum()),
        "tissue_mm2": round(float(tis.sum() * (f * f) * px_area / 1e6), 3),
        "mpp_analysis": round(mpp, 4), "downsample": round(ds, 3),
    }
    for p_ in IPTCR_PERCENTS:
        res[f"iptcr_pct{p_}"] = (counted & (pct >= p_)).sum() / n_count if n_count else float("nan")

    # ---- layers (level-0 coordinates)
    lut = lambda m: m[lab]
    layers = {
        "glomerulus": contours_of(glom, ds),
        "artery": contours_of(art, ds),
        "ptc": contours_of(lut(counted), ds),
        "dab_inflamed": contours_of(dab_infl, ds),
        "dab_normal": contours_of(dab_norm, ds),
    }

    if qc_prefix:
        save_qc(reader, thumb, f, lab, counted, inflamed, glom, art, leuko, qc_prefix, cen)
    # ---- one row per detected PTC (for the per-slide Excel table)
    details = []
    if SAVE_EXCEL:
        for i in np.flatnonzero(valid):
            status = ("inflamed" if inflamed[i] else "counted" if counted[i] else
                      "excluded: glomerulus/artery" if excl_ga[i] else
                      "excluded: inside DAB aggregate" if excl_aggr[i] else "excluded: outside ROI")
            details.append({"PTC #": int(i), "Status": status,
                            "X (level 0 px)": round(float(cen[i, 0] * ds), 1),
                            "Y (level 0 px)": round(float(cen[i, 1] * ds), 1),
                            "Area (um2)": round(float(area[i] * px_area), 1),
                            "Leukocyte pixels touching": int(leuko_ring[i]),
                            "Brown % of PTC area": round(float(pct[i]), 1)})
    return res, layers, details

In [99]:
# ------------------------------------------------------------------- outputs
def save_xml(layers, path, mpp0):
    root = ET.Element("Annotations", MicronsPerPixel=str(mpp0 or ""))
    for i, (name, polys) in enumerate(layers.items(), 1):
        ann = ET.SubElement(root, "Annotation", {"Id": str(i), "Name": LAYER_PREFIX + name,
                                                 "LineColor": str(LAYERS[name][0]), "Type": "4",
                                                 "Visible": "1", "ReadOnly": "0"})
        attrs = ET.SubElement(ann, "Attributes")
        ET.SubElement(attrs, "Attribute", {"Name": LAYER_PREFIX + name, "Id": "0", "Value": ""})
        regions = ET.SubElement(ann, "Regions")
        for r, p in enumerate(polys, 1):
            reg = ET.SubElement(regions, "Region", {"Id": str(r), "Type": "0", "NegativeROA": "0"})
            vs = ET.SubElement(reg, "Vertices")
            for x, y in list(p) + [p[0]]:
                ET.SubElement(vs, "Vertex", {"X": f"{x:.1f}", "Y": f"{y:.1f}", "Z": "0"})
    ET.ElementTree(root).write(path, encoding="UTF-8", xml_declaration=True)
 
 
def save_qc(reader, thumb, f, lab, counted, inflamed, glom, art, leuko, prefix, cen):
    """Overview + zoomed crops with outlines: PTC green, inflamed red, glom blue, artery magenta, DAB yellow."""
    def small(mask):
        return cv.resize(mask.astype(np.uint8) * 255, (thumb.shape[1], thumb.shape[0]),
                         interpolation=cv.INTER_AREA) > 20
    ov = thumb.copy()
    for m, col in ((counted[lab] & ~inflamed[lab], (0, 200, 0)), (glom, (0, 0, 255)),
                   (art, (255, 0, 255)), (inflamed[lab], (255, 0, 0))):
        ov[small(m)] = col
    cv.imwrite(prefix + "_overview.png", cv.cvtColor(np.concatenate([thumb, ov], 1), cv.COLOR_RGB2BGR))
 
    ids = np.flatnonzero(counted)
    rng = np.random.default_rng(0)
    size = 768
    for k, i in enumerate(rng.choice(ids, size=min(N_QC_CROPS, len(ids)), replace=False) if len(ids) else []):
        x = int(np.clip(cen[i, 0] - size // 2, 0, max(reader.W - size, 0)))
        y = int(np.clip(cen[i, 1] - size // 2, 0, max(reader.H - size, 0)))
        img = reader.read(x, y, size)
        out = img.copy()
        sl = (slice(y, y + size), slice(x, x + size))
        for m, col in ((counted[lab[sl]] & ~inflamed[lab[sl]], (0, 200, 0)), (inflamed[lab[sl]], (255, 0, 0)),
                       (glom[sl], (0, 0, 255)), (art[sl], (255, 0, 255)), (leuko[sl], (255, 255, 0))):
            cs, _ = cv.findContours(m.astype(np.uint8), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
            cv.drawContours(out, cs, -1, col, 1)
        cv.imwrite(f"{prefix}_crop{k + 1}.png", cv.cvtColor(np.concatenate([img, out], 1), cv.COLOR_RGB2BGR))
 
 
def results_dir():
    d = os.path.join(WORK_DIR, "results")
    os.makedirs(d, exist_ok=True)
    return d
 
 
def write_combined_csv():
    rows = []
    for fn in sorted(os.listdir(results_dir())):
        if fn.endswith(".json"):
            with open(os.path.join(results_dir(), fn)) as fh:
                rows.append(json.load(fh))
    rows.sort(key=lambda r: (r.get("folder", ""), r.get("slide", "")))
    path = os.path.join(WORK_DIR, "iptcr_results.csv")
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=RESULT_FIELDS, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    if SAVE_EXCEL:
        try:
            write_results_excel(rows)
        except Exception as e:                                   # e.g. openpyxl not installed
            log(f"   Excel not written ({type(e).__name__}: {e}); the CSV has the same numbers")
    return path, rows
 
 
READABLE = {
    "slide": "Slide", "slide_key": "DeID / key", "folder": "Folder", "item_id": "Athena item ID",
    "status": "Status", "iptcr": "iptcr (>=1 leukocyte)", "n_ptc_counted": "PTCs counted",
    "n_ptc_inflamed": "PTCs inflamed", "n_ptc_detected": "PTCs detected",
    "n_ptc_excl_glom_artery": "PTCs excluded: glom/artery", "n_ptc_excl_dab_aggregate": "PTCs excluded: DAB aggregate",
    "n_ptc_excl_roi": "PTCs excluded: outside ROI", "n_glomeruli": "Glomeruli", "n_arteries": "Arteries",
    "n_leukocytes": "Leukocytes (DAB)", "n_dab_aggregates": "DAB aggregates", "tissue_mm2": "Tissue area (mm2)",
    "mpp_analysis": "Analysis um/px", "downsample": "Downsample", "minutes": "Minutes", "error": "Error",
}
READABLE.update({f"iptcr_pct{p}": f"iptcr (>= {p}% brown)" for p in IPTCR_PERCENTS})
 
 
def tidy_sheets(xw):
    """Readable column widths + frozen header row. Cosmetic only: never fails the run."""
    try:
        for ws in xw.sheets.values():
            for col in ws.iter_cols(max_row=200):
                width = max(len(str(c.value or "")) for c in col)
                ws.column_dimensions[col[0].column_letter].width = min(max(10, width + 2), 60)
            ws.freeze_panes = "A2"
    except Exception:
        pass
 
 
def write_results_excel(rows):
    import pandas as pd
    df = pd.DataFrame(rows, columns=RESULT_FIELDS)
    ok = df[df["status"] == "ok"].drop(columns=["status", "error"])
    for c in ["iptcr"] + [f"iptcr_pct{p}" for p in IPTCR_PERCENTS]:
        ok[c] = pd.to_numeric(ok[c], errors="coerce").round(4)
    err = df[df["status"] != "ok"][["slide", "item_id", "folder", "error"]]
    settings = pd.DataFrame([(k, str(v)) for k, v in {
        "Inflamed rule": f">= {MIN_LEUKO_PIXELS} leukocyte pixels inside or within {PTC_RING_PX} px of the PTC",
        "PTC model": PTC_MODEL_PATH, "Glom/artery model": GLOM_MODEL_PATH if USE_GLOM_MODEL else "not used",
        "Analysis resolution (um/px)": round(TRAIN_MPP, 4), "Tile / overlap": f"{TILE} px / {int(OVERLAP * 100)}%",
        "PTC threshold": PTC_THRESHOLD, "PTC min area (um2)": PTC_MIN_AREA_UM2,
        "Max PTC overlap with glom/artery": PTC_MAX_GLOM_ARTERY_OVERLAP,
        "DAB threshold": DAB_THRESHOLD, "Leukocyte area (um2)": f"{DAB_MIN_AREA_UM2} - {DAB_AGGREGATE_AREA_UM2}",
        "ROI layer": ROI_LAYER_NAME or "none (all tissue)", "Stain vectors": json.dumps(STAIN_VECTORS),
    }.items()], columns=["Setting", "Value"])
    path = os.path.join(WORK_DIR, "iptcr_results.xlsx")
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        ok.rename(columns=READABLE).to_excel(xw, sheet_name="iptcr per slide", index=False)
        err.rename(columns=READABLE).to_excel(xw, sheet_name="Errors", index=False)
        settings.to_excel(xw, sheet_name="Settings", index=False)
        tidy_sheets(xw)
 
 
def save_slide_table(details, res, slide_name, path):
    import pandas as pd
    summary = pd.DataFrame([("Slide", slide_name)] + [(READABLE.get(k, k), v) for k, v in res.items()],
                           columns=["Item", "Value"])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        summary.to_excel(xw, sheet_name="Summary", index=False)
        pd.DataFrame(details).to_excel(xw, sheet_name="PTCs", index=False)
        tidy_sheets(xw)
 
 

In [100]:
# ------------------------------------------------------------------- actions
def do_check(gc, slides):
    done = {}
    for fn in os.listdir(results_dir()):
        if fn.endswith(".json"):
            with open(os.path.join(results_dir(), fn)) as fh:
                r = json.load(fh)
            done[r["item_id"]] = r
    ref = {}
    if REFERENCE_CSV:
        import pandas as pd
        df = pd.read_csv(REFERENCE_CSV)
        idc = REFERENCE_ID_COLUMN or df.columns[0]
        cols = REFERENCE_SCORE_COLUMNS or [c for c in df.columns if "iptcr" in str(c).lower()]
        for _, r in df.iterrows():
            vals = {c: r[c] for c in cols if str(r[c]).strip() not in ("", "nan")}
            if vals:
                ref[slide_key(str(r[idc]))] = vals
        print(f"Reference CSV: {len(ref)} slides with scores in columns {cols}")
 
    rows = []
    for it, folder in slides:
        meta = {k: v for k, v in (it.get("meta") or {}).items() if "iptcr" in k.lower()}
        anns = gc.get("annotation", parameters={"itemId": it["_id"], "limit": 0})
        ai = sorted({a["annotation"].get("name", "") for a in anns
                     if a["annotation"].get("name", "").startswith(LAYER_PREFIX)})
        r = done.get(it["_id"], {})
        key = slide_key(it["name"])
        rows.append({"slide": it["name"], "slide_key": key, "folder": folder, "item_id": it["_id"],
                     "this_pipeline_iptcr": r.get("iptcr", "") if r.get("status") == "ok" else "",
                     "athena_metadata_iptcr": json.dumps(meta) if meta else "",
                     "athena_AI_layers": len(ai),
                     "reference_csv_iptcr": json.dumps(ref.get(key, {}), default=str) if key in ref else ""})
    out = os.path.join(WORK_DIR, "check_existing_scores.csv")
    with open(out, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()) if rows else ["slide"])
        w.writeheader()
        w.writerows(rows)
 
    has = lambda r: r["this_pipeline_iptcr"] != "" or r["athena_metadata_iptcr"] or r["reference_csv_iptcr"]
    print(f"\n{len(rows)} slides. With an existing iptcr score: {sum(1 for r in rows if has(r))}")
    print(f"   from this pipeline: {sum(1 for r in rows if r['this_pipeline_iptcr'] != '')}   "
          f"in Athena metadata: {sum(1 for r in rows if r['athena_metadata_iptcr'])}   "
          f"in reference CSV: {sum(1 for r in rows if r['reference_csv_iptcr'])}   "
          f"with AI layers on Athena: {sum(1 for r in rows if r['athena_AI_layers'])}")
    print("\nSlides with NO iptcr anywhere:")
    for r in rows:
        if not has(r):
            print(f"   {r['slide_key']:14} {r['slide']}")
    print(f"\nFull table: {out}")
 
 
def do_upload(gc, slides):
    plan = []
    for it, folder in slides:
        xml_path = os.path.join(WORK_DIR, "xml", file_base(it["name"]) + ".xml")
        if os.path.exists(xml_path):
            plan.append((it, xml_path))
        else:
            log(f"   not run yet (no XML), skipped: {it['name']}")
    log(f"{len(plan)} slides have results to upload to {INSTANCE}")
    if not plan:
        return
    print("Layers added to each slide: " + ", ".join(LAYER_PREFIX + n for n in LAYERS))
    if REPLACE_AI_LAYERS:
        print(f"Existing layers with these exact names on those slides are replaced (other layers untouched).")
    if not ASSUME_YES:
        try:
            answer = input(f"\nThis adds annotation layers to {len(plan)} slide(s) on {INSTANCE}. "
                           f"Type YES to continue: ").strip()
        except EOFError:                       # batch job: nobody can type
            answer = ""
            print("\nNo keyboard (batch job). Add --yes to the command to confirm the upload.")
        if answer != "YES":
            print("Stopped. Nothing uploaded.")
            return
    for k, (it, xml_path) in enumerate(plan, 1):
        try:
            layers = read_xml_layers(xml_path)
            upload_layers(gc, it["_id"], layers)
            if WRITE_METADATA:
                rp = os.path.join(results_dir(), it["_id"] + ".json")
                if os.path.exists(rp):
                    with open(rp) as fh:
                        r = json.load(fh)
                    if r.get("status") == "ok":
                        gc.addMetadataToItem(it["_id"], {"AI_iptcr": round(float(r["iptcr"]), 4),
                                                         "AI_ptc_counted": r["n_ptc_counted"],
                                                         "AI_ptc_inflamed": r["n_ptc_inflamed"]})
            log(f"[{k}/{len(plan)}] {it['name']}: " +
                ", ".join(f"{n} {len(p)}" for n, p in layers.items() if p))
        except Exception as e:
            log(f"[{k}/{len(plan)}] {it['name']}: ERROR {type(e).__name__}: {e}")
    log("Upload finished. Open a slide in HistomicsUI and look for the layers starting with "
        f"'{LAYER_PREFIX.strip()}'.")
 
 
def do_run(gc, slides):
    os.makedirs(WORK_DIR, exist_ok=True)
    for sub in ("xml", "qc", "tables"):
        os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)
    todo = []
    for it, folder in slides:
        rp = os.path.join(results_dir(), it["_id"] + ".json")
        if not RERUN and os.path.exists(rp):
            with open(rp) as fh:
                if json.load(fh).get("status") == "ok":
                    continue
        todo.append((it, folder))
    log(f"{len(slides)} slides, {len(slides) - len(todo)} already done, {len(todo)} to process")
    if not todo:
        write_combined_csv()
        return
 
    ptc_model, glom_model = load_models()
    for k, (it, folder) in enumerate(todo, 1):
        t0 = time.time()
        name = it["name"]
        base = file_base(name)
        log(f"[{k}/{len(todo)}] {name}")
        res = {"slide": name, "slide_key": slide_key(name), "item_id": it["_id"], "folder": folder}
        path = None
        try:
            roi = None
            if ROI_LAYER_NAME:
                roi = roi_polygons(gc, it["_id"])
                if not roi:
                    raise RuntimeError(f"no ROI layer named {ROI_LAYER_NAME!r} on this slide")
            path = download_slide(gc, it)
            reader = SlideReader(path)
            qc = os.path.join(WORK_DIR, "qc", base) if SAVE_QC_IMAGES else None
            out, layers, details = analyse(reader, ptc_model, glom_model, roi, qc)
            if SAVE_EXCEL:
                try:
                    save_slide_table(details, out, name, os.path.join(WORK_DIR, "tables", base + ".xlsx"))
                except Exception as e:                           # a table problem must not fail the slide
                    log(f"   per-PTC Excel not written ({type(e).__name__}: {e})")
            res.update(out)
            if SAVE_XML:
                save_xml(layers, os.path.join(WORK_DIR, "xml", base + ".xml"), reader.mpp0)
            if UPLOAD_TO_ATHENA:
                upload_layers(gc, it["_id"], layers)
            if WRITE_METADATA:
                gc.addMetadataToItem(it["_id"], {"AI_iptcr": round(float(out["iptcr"]), 4),
                                                 "AI_ptc_counted": out["n_ptc_counted"],
                                                 "AI_ptc_inflamed": out["n_ptc_inflamed"]})
            res["status"] = "ok"
            log(f"   iptcr {out['iptcr']:.3f}  ({out['n_ptc_inflamed']}/{out['n_ptc_counted']} PTCs)  "
                f"glomeruli {out['n_glomeruli']}  arteries {out['n_arteries']}")
        except Exception as e:
            res["status"] = "error"
            res["error"] = f"{type(e).__name__}: {e}"
            log("   ERROR " + res["error"])
            traceback.print_exc()
        finally:
            res["minutes"] = round((time.time() - t0) / 60, 1)
            with open(os.path.join(results_dir(), it["_id"] + ".json"), "w") as fh:
                json.dump(res, fh, default=float)
            if DELETE_SLIDES_AFTER and path and os.path.exists(path):
                os.remove(path)
    out, rows = write_combined_csv()
    ok = [r for r in rows if r.get("status") == "ok"]
    log(f"Done. {len(ok)} slides with results, {len(rows) - len(ok)} errors. Results: {out}")
 

In [101]:

def main():
    global ACTION, ITEM_ID, FOLDER_IDS, ASSUME_YES
    ap = argparse.ArgumentParser()
    ap.add_argument("--action", choices=["check", "run", "upload"])
    ap.add_argument("--item-id")
    ap.add_argument("--folder-id", action="append")
    ap.add_argument("--shard", help="k/n: process every n-th slide starting at k (SLURM job arrays)")
    ap.add_argument("--yes", action="store_true", help="upload without asking (batch jobs)")
    # parse_known_args: ignore arguments Jupyter adds by itself (e.g. "-f kernel.json"),
    # so the script also works when pasted into a notebook cell
    a, _ = ap.parse_known_args()
    ACTION = a.action or ACTION
    ASSUME_YES = ASSUME_YES or a.yes
    if a.item_id:
        ITEM_ID = a.item_id
    if a.folder_id:
        FOLDER_IDS, ITEM_ID = a.folder_id, None
    if ITEM_ID and "PASTE" in ITEM_ID:
        raise SystemExit("Paste a slide item ID into ITEM_ID, or comment that line out to use FOLDER_IDS.")
    if not ITEM_ID and not FOLDER_IDS:
        raise SystemExit("Set FOLDER_IDS (or ITEM_ID).")
 
    gc = connect()
    slides = list_slides(gc, ITEM_ID, FOLDER_IDS)
    if a.shard:
        k, n = map(int, a.shard.split("/"))
        slides = slides[k::n]
    log(f"{'TEST: 1 slide' if ITEM_ID else f'{len(slides)} slides'}  ACTION={ACTION}")
    {"check": do_check, "run": do_run, "upload": do_upload}[ACTION](gc, slides)
 
 
if __name__ == "__main__":
    main()

Login or email:  praneethvempati
Password for praneethvempati:  ········


17:37:20 11 slides  ACTION=run
17:37:20 11 slides, 0 already done, 11 to process
17:37:20 GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
17:37:27 [1/11] JHH_DS_1 CD34LCA.svs
17:37:27    downloading JHH_DS_1 CD34LCA.svs (314 MB)
17:37:28    analysis size 6977x14443 px at 0.912 um/px, 1203 PTC tiles
17:37:31    PTC tiles 16/1203
17:38:04    PTC tiles 816/1203
17:38:22    glom tiles: 301 x 512 px
17:40:04    iptcr 0.636  (2335/3669 PTCs)  glomeruli 16  arteries 17
17:40:04 [2/11] JHH_DS_10 CD34LCA.svs
17:40:04    downloading JHH_DS_10 CD34LCA.svs (371 MB)
17:40:06    analysis size 8050x14928 px at 0.912 um/px, 874 PTC tiles
17:40:06    PTC tiles 16/874
17:40:39    PTC tiles 816/874
17:40:42    glom tiles: 245 x 512 px
17:42:05    iptcr 0.389  (1289/3315 PTCs)  glomeruli 11  arteries 5
17:42:05 [3/11] JHH_DS_11 CD34LCA.svs
17:42:05    downloading JHH_DS_11 CD34LCA.svs (288 MB)
17:42:06    analysis size 8587x10964 px at 0.912 um/px, 603 PTC tiles
17:42:07    PTC ti